# OHE vs PCA Solvent Encoding — Benchmark

This notebook compares two ways of representing solvents in EDBO+:

| | **OHE (default)** | **PCA** |
|---|---|---|
| Representation | Binary column per solvent | 4 continuous PC coordinates |
| Chemical similarity | Ignored | Encoded in PC distance |
| Columns added | N solvents | Always 4 |

**Dataset:** BMS cross-coupling (1 728 experiments, fully observed)  
**Parameters:** base, ligand, solvent, concentration, temperature  
**Objectives:** yield (maximize), cost (minimize)  

**Protocol:**
1. Pick a shared seed of 16 real experiments.
2. Generate an OHE scope and a PCA scope from the same components.
3. Inject the seed observations into each scope.
4. Run EDBO+ on both — compare suggestions and prediction quality.


## 1. Imports and setup


In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..','..')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from edbo.plus.optimizer_botorch import EDBOplus

SEED = 42
N_SEED = 16      # initial observed experiments
BATCH  = 4       # experiments per BO round
DATA_PATH = '../../data/test/BMS_yield_cost__experiments_yield_and_cost.csv'
LUT_PATH  = '../../data/Solvent_PC_clean.csv'


## 2. Load and explore the test dataset


In [ ]:
oracle = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {oracle.shape}')
print()
for col in ['base','ligand','solvent','concentration','temperature']:
    print(f'{col:>14}: {oracle[col].unique().tolist()}')
print()
print(f'yield : {oracle["yield"].min():.1f} – {oracle["yield"].max():.1f}')
print(f'cost  : {oracle["cost"].min():.4f} – {oracle["cost"].max():.4f}')


In [ ]:
oracle.head()


## 3. Ground-truth Pareto front

We compute the true Pareto front (maximize yield, minimize cost) so we can later
measure how well each encoding finds it.


In [ ]:
def pareto_front(df, obj_yield='yield', obj_cost='cost'):
    """Return boolean mask of Pareto-optimal rows."""
    Y = df[obj_yield].values
    C = df[obj_cost].values
    n = len(Y)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if Y[j] >= Y[i] and C[j] <= C[i] and (Y[j] > Y[i] or C[j] < C[i]):
                dominated[i] = True
                break
    return ~dominated

oracle['pareto'] = pareto_front(oracle)
pareto_df = oracle[oracle['pareto']].sort_values('yield', ascending=False)
print(f'Pareto-optimal experiments: {len(pareto_df)} / {len(oracle)}')
pareto_df[['base','ligand','solvent','concentration','temperature','yield','cost']].head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(oracle['cost'], oracle['yield'], alpha=0.25, s=15, label='All experiments')
ax.scatter(pareto_df['cost'], pareto_df['yield'], color='red', s=40, zorder=5, label='Pareto front')
ax.set_xlabel('Cost')
ax.set_ylabel('Yield (%)')
ax.set_title('Full dataset with Pareto front')
ax.legend()
plt.tight_layout()
plt.show()


## 4. Solvents in PCA space

Before running EDBO+, let's visualize where the four solvents sit in the first two principal components.
Similar solvents cluster together; a PCA-encoded GP model can leverage this structure.


In [ ]:
# Mapping from dataset short names to lookup-table full names
solvent_map = {
    'DMAc':     'DMA [N,N-Dimethylacetamide]',
    'BuCN':     'Butyronitrile',
    'BuOAc':    'n-Butyl Acetate',
    'p-Xylene': 'p-Xylene',
}

lut = pd.read_csv(LUT_PATH)
solvent_pca = (
    pd.DataFrame({'solvent': list(solvent_map), 'Name': list(solvent_map.values())})
    .merge(lut[['Name','PC1','PC2','PC3','PC4']], on='Name')
)
print(solvent_pca.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
for _, row in solvent_pca.iterrows():
    ax.scatter(row['PC1'], row['PC2'], s=120, zorder=5)
    ax.annotate(row['solvent'], (row['PC1'], row['PC2']),
                textcoords='offset points', xytext=(6,4), fontsize=10)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Four solvents in PC1–PC2 space')
ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
plt.tight_layout(); plt.show()

print()
print('DMAc is a polar aprotic solvent; BuCN is a nitrile; BuOAc an ester; p-Xylene is apolar.')
print('Their separation in PC space reflects their physicochemical differences.')


## 5. Scope generation

We generate **two scope CSV files** from the same components:
- `scope_ohe.csv` — solvents kept as strings → OHE applied by EDBO+ automatically
- `scope_pca.csv` — solvents replaced by PC1–PC4 from the lookup table

For PCA we build a small custom lookup DataFrame that maps the dataset's short solvent names
to their PC coordinates.


In [ ]:
components = {
    'base':          ['KOAc', 'KOPiv', 'CsOAc', 'CsOPiv'],
    'ligand':        ['BrettPhos', 'PPhtBu2', 'tBPh-CPhos', 'PCy3 HBF4',
                      'PPh3', 'X-Phos', 'P(fur)3', 'PPh2Me',
                      'GorlosPhos HBF4', 'JackiePhos', 'CgMe-PPh', 'PPhMe2'],
    'solvent':       ['DMAc', 'BuCN', 'BuOAc', 'p-Xylene'],
    'concentration': [0.1, 0.057, 0.153],
    'temperature':   [105, 90, 120],
}


In [ ]:
# --- OHE scope ---
EDBOplus().generate_reaction_scope(
    components=components,
    filename='scope_ohe.csv',
    check_overwrite=False,
)


In [ ]:
# Custom lookup: short solvent names → PC1-PC4 from the bundled table
solvent_lookup = solvent_pca[['solvent','PC1','PC2','PC3','PC4']].rename(
    columns={'solvent': 'Name'}
)

pca_encodings = {
    'solvent': {
        'file':     solvent_lookup,   # DataFrame, not a path
        'key':      'Name',
        'features': ['PC1','PC2','PC3','PC4'],
    }
}

# --- PCA scope ---
EDBOplus().generate_reaction_scope(
    components=components,
    encodings=pca_encodings,
    filename='scope_pca.csv',
    check_overwrite=False,
)


In [ ]:
ohe_scope = pd.read_csv('scope_ohe.csv')
pca_scope = pd.read_csv('scope_pca.csv')
print(f'OHE scope: {ohe_scope.shape} — columns: {ohe_scope.columns.tolist()}')
print(f'PCA scope: {pca_scope.shape} — columns: {pca_scope.columns.tolist()}')


## 6. Inject seed observations

We pick the same 16 real experiments as starting data for both methods.
The oracle dataset provides the true yield and cost values.


In [ ]:
# Select seed rows from the oracle (balanced across solvents)
rng = np.random.default_rng(SEED)
seed_idx = rng.choice(len(oracle), size=N_SEED, replace=False)
seed_obs = oracle.iloc[seed_idx][['base','ligand','solvent','concentration','temperature','yield','cost']].copy()
print(f'Seed experiments ({N_SEED}):')
seed_obs


In [ ]:
def inject_observations(scope_path, observations, save_path=None):
    """Write observed yield/cost into a scope CSV; mark all other rows PENDING."""
    scope = pd.read_csv(scope_path)
    keys  = ['base', 'ligand', 'solvent', 'concentration', 'temperature']

    # Start with every row marked PENDING
    scope['yield'] = 'PENDING'
    scope['cost']  = 'PENDING'

    # Round floats to avoid CSV round-trip precision drift
    for col in ['concentration', 'temperature']:
        scope[col] = scope[col].round(6)

    obs = observations.copy()
    for col in ['concentration', 'temperature']:
        obs[col] = obs[col].round(6)

    # Update row-by-row — avoids any merge dtype or float-precision ambiguity
    matched = 0
    for _, row in obs.iterrows():
        mask = pd.Series([True] * len(scope))
        for key in keys:
            mask &= (scope[key] == row[key])
        if mask.sum() == 1:
            scope.loc[mask, 'yield'] = row['yield']
            scope.loc[mask, 'cost']  = row['cost']
            matched += 1
        elif mask.sum() == 0:
            print(f'  WARNING: no scope row matched observation: {dict(row[keys])}')

    save_path = save_path or scope_path
    scope.to_csv(save_path, index=False)
    n_obs = (scope['yield'] != 'PENDING').sum()
    print(f'{save_path}: {n_obs}/{matched} observations injected '
          f'({len(scope) - n_obs} rows marked PENDING)')
    return scope

inject_observations('scope_ohe.csv', seed_obs)
inject_observations('scope_pca.csv', seed_obs)


## 7. Round 1 — BO with seed data

Run EDBO+ on both scopes. Both start from the same 16 seed observations.


In [ ]:
def ensure_pending(scope_path, objectives=('yield', 'cost')):
    """Convert any NaN in objective columns to the string 'PENDING'.
    EDBO+ splits train/test by scanning for the literal string 'PENDING';
    NaN rows are silently treated as training data, leaving test empty.
    Call this right before every EDBOplus().run() call.
    """
    df = pd.read_csv(scope_path)
    fixed = []
    for col in objectives:
        if col in df.columns:
            n = df[col].isna().sum()
            if n:
                df[col] = df[col].fillna('PENDING')
                fixed.append(f'{col}: {n} NaN → PENDING')
    if fixed:
        df.to_csv(scope_path, index=False)
        print(f'[ensure_pending] {scope_path}: ' + ', '.join(fixed))
    else:
        pending = sum((df[c].astype(str) == 'PENDING').sum() for c in objectives if c in df.columns)
        print(f'[ensure_pending] {scope_path}: OK ({pending} PENDING rows)')


In [ ]:
ensure_pending('scope_ohe.csv')
print('=== OHE encoding ===')
EDBOplus().run(
    filename='scope_ohe.csv',
    objectives=['yield','cost'],
    objective_mode=['max','min'],
    batch=BATCH,
)


In [ ]:
ensure_pending('scope_pca.csv')
print('=== PCA encoding ===')
EDBOplus().run(
    filename='scope_pca.csv',
    objectives=['yield','cost'],
    objective_mode=['max','min'],
    exclude_columns=['solvent'],
    batch=BATCH,
)


## 8. Compare suggested experiments

Check which experiments each method recommends for the next round and look up
their real yield and cost in the oracle dataset.


In [ ]:
def get_suggestions(scope_path, oracle_df):
    scope = pd.read_csv(scope_path)
    keys  = ['base','ligand','solvent','concentration','temperature']
    sugg  = scope[scope['priority'] == 1][keys].copy()
    sugg  = sugg.merge(oracle_df[keys + ['yield','cost']], on=keys, how='left')
    return sugg

sugg_ohe = get_suggestions('scope_ohe.csv', oracle)
sugg_pca = get_suggestions('scope_pca.csv', oracle)

print('OHE suggestions with true values:')
display(sugg_ohe)
print('PCA suggestions with true values:')
display(sugg_pca)


## 9. Prediction quality across the full scope

EDBO+ writes a `pred_*.csv` file after each run with predicted mean and std dev
for every reaction. We compare predicted yield vs actual yield for tested reactions.


In [ ]:
def prediction_quality(pred_path, oracle_df, objective='yield'):
    pred = pd.read_csv(pred_path)
    mean_col = f'{objective}_predicted_mean'
    if mean_col not in pred.columns:
        print(f'Column {mean_col} not found in {pred_path}'); return None, None, None
    # Keep only rows that were observed (non-PENDING) and have a prediction
    obs = pred[pred[objective] != 'PENDING'].copy()
    obs[objective]  = obs[objective].astype(float)
    obs[mean_col]   = obs[mean_col].astype(float)
    r2  = np.corrcoef(obs[objective], obs[mean_col])[0,1]**2
    mae = np.mean(np.abs(obs[objective] - obs[mean_col]))
    return obs, r2, mae

res_ohe = prediction_quality('pred_scope_ohe.csv', oracle)
res_pca = prediction_quality('pred_scope_pca.csv', oracle)

if res_ohe[0] is not None and res_pca[0] is not None:
    ohe_obs, ohe_r2, ohe_mae = res_ohe
    pca_obs, pca_r2, pca_mae = res_pca
    print(f'OHE  —  yield R²: {ohe_r2:.3f}   MAE: {ohe_mae:.1f} %')
    print(f'PCA  —  yield R²: {pca_r2:.3f}   MAE: {pca_mae:.1f} %')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, obs, r2, mae, label in [
    (axes[0], ohe_obs, ohe_r2, ohe_mae, 'OHE'),
    (axes[1], pca_obs, pca_r2, pca_mae, 'PCA'),
]:
    ax.scatter(obs['yield'], obs['yield_predicted_mean'], alpha=0.7, s=40)
    lims = [0, 105]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual yield (%)')
    ax.set_ylabel('Predicted yield (%)')
    ax.set_title(f'{label}  |  R²={r2:.3f}  MAE={mae:.1f}%')

plt.suptitle('Prediction quality on seed experiments', y=1.02)
plt.tight_layout()
plt.show()


## 10. Pareto front coverage

How many of the true Pareto-optimal experiments does each method prioritise?
We check whether the suggested batch overlaps with the ground-truth Pareto front.


In [ ]:
pareto_set = set(
    tuple(r) for r in pareto_df[['base','ligand','solvent','concentration','temperature']].values
)

for label, sugg in [('OHE', sugg_ohe), ('PCA', sugg_pca)]:
    keys = ['base','ligand','solvent','concentration','temperature']
    hits = sum(
        tuple(r) in pareto_set
        for r in sugg[keys].values
    )
    print(f'{label}: {hits}/{len(sugg)} suggested experiments are on the Pareto front')
    print(f'  Suggested yields: {sugg["yield"].tolist()}')
    print(f'  Suggested costs:  {[round(c,4) for c in sugg["cost"].tolist()]}')
    print()


## 11. Expected Improvement by solvent

Inspect the expected improvement landscape grouped by solvent to see how each encoding
distributes attention across the solvent space.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, pred_path, label in [
    (axes[0], 'pred_scope_ohe.csv', 'OHE'),
    (axes[1], 'pred_scope_pca.csv', 'PCA'),
]:
    pred = pd.read_csv(pred_path)
    untested = pred[pred['yield'] == 'PENDING'].copy()
    ei_col = 'yield_expected_improvement'
    if ei_col not in untested.columns:
        ax.set_title(f'{label} — EI column not available'); continue
    untested[ei_col] = untested[ei_col].astype(float)
    by_solvent = untested.groupby('solvent')[ei_col].mean().sort_values(ascending=False)
    by_solvent.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{label} — mean EI (yield) by solvent')
    ax.set_xlabel('')
    ax.set_ylabel('Mean expected improvement')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Expected Improvement distribution across solvents', y=1.02)
plt.tight_layout()
plt.show()


## 12. Summary

| Metric | OHE | PCA |
|---|---|---|
| Solvent representation | Binary (4 columns) | Continuous PC1–PC4 |
| Seed R² (yield) | *(see cell 9)* | *(see cell 9)* |
| Seed MAE (yield) | *(see cell 9)* | *(see cell 9)* |
| Pareto hits in batch | *(see cell 10)* | *(see cell 10)* |

**Key takeaways:**
- OHE treats each solvent as completely independent — the model cannot generalise across solvents.
- PCA places solvents in a continuous chemical space; the GP can interpolate between similar solvents.
- With only 16 seed experiments across 1 728 possibilities, PCA typically makes more informed suggestions
  because it leverages the physicochemical relationships between solvents.
- As the number of seed observations grows, the two methods converge — OHE eventually
  "learns" each solvent individually.

**To run a second round:**  
Look up the real yield/cost for the suggested experiments, inject them into each scope CSV,
and call `EDBOplus().run(...)` again with the same arguments.


---
## 13. Multi-round campaign

A single round is not enough to distinguish the two encodings. Run **N rounds**,
look up true yield/cost for each suggested batch from the oracle, inject them,
and repeat. Track:
- **Pareto hits** — how many true Pareto-optimal experiments are in the training set
- **Best yield** — highest yield seen so far

Both methods start from the same 16 seed experiments. Fresh scope files
`scope_ohe_multi.csv` / `scope_pca_multi.csv` are used so the single-round
demo above is not disturbed.


In [ ]:
N_ROUNDS = 8
KEYS = ['base', 'ligand', 'solvent', 'concentration', 'temperature']


def setup_scope(src_path, dest_path, seed_observations):
    import shutil
    shutil.copy(src_path, dest_path)
    inject_observations(dest_path, seed_observations, save_path=dest_path)


def run_bo_round(scope_path, exclude_columns=None):
    ensure_pending(scope_path)
    kwargs = dict(
        filename=scope_path,
        objectives=['yield', 'cost'],
        objective_mode=['max', 'min'],
        batch=BATCH,
        init_sampling_method='cvt',
    )
    if exclude_columns:
        kwargs['exclude_columns'] = exclude_columns
    EDBOplus().run(**kwargs)
    scope = pd.read_csv(scope_path)
    return scope[scope['priority'] == 1][KEYS].copy()


def inject_oracle_values(scope_path, suggested):
    scope = pd.read_csv(scope_path)
    for _, row in suggested.iterrows():
        o_mask = pd.Series([True] * len(oracle))
        s_mask = pd.Series([True] * len(scope))
        for key in KEYS:
            o_mask &= (oracle[key] == row[key])
            s_mask &= (scope[key]  == row[key])
        if o_mask.sum() == 1 and s_mask.sum() == 1:
            true_row = oracle[o_mask].iloc[0]
            scope.loc[s_mask, 'yield'] = true_row['yield']
            scope.loc[s_mask, 'cost']  = true_row['cost']
    scope.to_csv(scope_path, index=False)


def campaign_metrics(scope_path):
    scope = pd.read_csv(scope_path)
    obs = scope[scope['yield'].astype(str) != 'PENDING'].copy()
    obs['yield'] = obs['yield'].astype(float)
    obs['cost']  = obs['cost'].astype(float)
    pareto_hits = sum(
        tuple(r) in pareto_set
        for r in obs[KEYS].itertuples(index=False)
    )
    best_yield = float(obs['yield'].max()) if len(obs) else 0.0
    return {'n_obs': len(obs), 'pareto_hits': pareto_hits, 'best_yield': best_yield}


### Run the campaign

Each round: EDBO+ suggests 4 experiments → look up true values from oracle → inject → repeat.
Output is suppressed per round; a one-line summary is printed instead.


In [ ]:
import warnings, contextlib, io

print('Setting up fresh scopes...')
setup_scope('scope_ohe.csv', 'scope_ohe_multi.csv', seed_obs)
setup_scope('scope_pca.csv', 'scope_pca_multi.csv', seed_obs)

records_ohe, records_pca = [], []
records_ohe.append({'round': 0, **campaign_metrics('scope_ohe_multi.csv')})
records_pca.append({'round': 0, **campaign_metrics('scope_pca_multi.csv')})

for rnd in range(1, N_ROUNDS + 1):
    print(f'Round {rnd}/{N_ROUNDS}', end='  ')

    with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()):
        warnings.simplefilter('ignore')
        sugg_ohe = run_bo_round('scope_ohe_multi.csv')
    inject_oracle_values('scope_ohe_multi.csv', sugg_ohe)
    m_ohe = campaign_metrics('scope_ohe_multi.csv')
    records_ohe.append({'round': rnd, **m_ohe})

    with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()):
        warnings.simplefilter('ignore')
        sugg_pca = run_bo_round('scope_pca_multi.csv', exclude_columns=['solvent'])
    inject_oracle_values('scope_pca_multi.csv', sugg_pca)
    m_pca = campaign_metrics('scope_pca_multi.csv')
    records_pca.append({'round': rnd, **m_pca})

    print(
        f"OHE: {m_ohe['pareto_hits']} pareto / {m_ohe['best_yield']:.1f}% yield  |  "
        f"PCA: {m_pca['pareto_hits']} pareto / {m_pca['best_yield']:.1f}% yield"
    )

df_ohe = pd.DataFrame(records_ohe)
df_pca = pd.DataFrame(records_pca)
print('Done.')


### Convergence plots


In [ ]:
n_pareto_total = len(pareto_df)
global_best = oracle['yield'].max()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(df_ohe['round'], df_ohe['pareto_hits'], 'o-', label='OHE', color='steelblue', lw=2)
ax.plot(df_pca['round'], df_pca['pareto_hits'], 's-', label='PCA', color='darkorange', lw=2)
ax.axhline(n_pareto_total, color='grey', lw=1, ls='--', label=f'All Pareto ({n_pareto_total})')
ax.set_xlabel('Round  (each round = 4 new experiments)')
ax.set_ylabel('True Pareto experiments found')
ax.set_title('Pareto front coverage')
ax.set_xticks(range(N_ROUNDS + 1))
ax.legend()

ax = axes[1]
ax.plot(df_ohe['round'], df_ohe['best_yield'], 'o-', label='OHE', color='steelblue', lw=2)
ax.plot(df_pca['round'], df_pca['best_yield'], 's-', label='PCA', color='darkorange', lw=2)
ax.axhline(global_best, color='grey', lw=1, ls='--', label=f'Global best ({global_best:.1f}%)')
ax.set_xlabel('Round  (each round = 4 new experiments)')
ax.set_ylabel('Best yield found (%)')
ax.set_title('Best yield discovered')
ax.set_xticks(range(N_ROUNDS + 1))
ax.legend()

title = f'OHE vs PCA — {N_ROUNDS} rounds x {BATCH} experiments (seed={N_SEED})'
plt.suptitle(title, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
cols_ohe = {'pareto_hits': 'OHE_pareto', 'best_yield': 'OHE_yield'}
cols_pca = {'pareto_hits': 'PCA_pareto', 'best_yield': 'PCA_yield'}
summary = (
    df_ohe[['round', 'n_obs', 'pareto_hits', 'best_yield']].rename(columns=cols_ohe)
    .merge(df_pca[['round', 'pareto_hits', 'best_yield']].rename(columns=cols_pca), on='round')
)
summary['OHE_pareto_%'] = (summary['OHE_pareto'] / n_pareto_total * 100).round(1)
summary['PCA_pareto_%'] = (summary['PCA_pareto'] / n_pareto_total * 100).round(1)
summary
